GROUP NAME: Cognitive Crew

TEAM MEMBERS:
  1. Lanka Devi Satwika - B24DS013
  2. Kotapati Sai Mounika - B24CS019
  3. Bailapudi Kusuma Teja - B24CS011
  4. Bodike Chaithali - B24CS013
  5. Jayasurya Boorada

## PS-3: Robot Strategy Battle

In this problem, two robots A and B compete to collect energy cells from a warehouse grid.

Robot A is MAX and tries to get a better score, while robot B is MIN and tries to reduce MAX's advantage.

We use Minimax and Alpha-Beta Pruning to find the best move for MAX.

In [30]:
import time

# Directions
DIRECTIONS = [
    (-1, 0, "UP"),
    (0, 1, "RIGHT"),
    (1, 0, "DOWN"),
    (0, -1, "LEFT")
]


# Check whether a position is inside the board
# and is not an obstacle.
def valid_position(row, col, board):
    rows = len(board)
    cols = len(board[0])

    if row < 0 or row >= rows:
        return False

    if col < 0 or col >= cols:
        return False

    if board[row][col] == '#':
        return False

    return True

In [31]:
# Generate possible moves for a robot.
def generate_moves(position, board):
    moves = []

    row, col = position

    for dr, dc, name in DIRECTIONS:
        new_row = row + dr
        new_col = col + dc

        if valid_position(new_row, new_col, board):
            new_position = (new_row, new_col)
            moves.append((name, new_position))

    return moves

### Generating Possible Moves

For each robot, we generate all possible moves in the given order:

UP, RIGHT, DOWN, LEFT.

Only valid moves are added to the list.

In [32]:
# Manhattan distance
def manhattan_distance(p1, p2):
    return abs(p1[0] - p2[0]) + abs(p1[1] - p2[1])

### Manhattan Distance

Manhattan distance is used to find how close a robot is to an energy cell.

It is calculated using the row and column differences.

In [33]:
# Evaluation function
def evaluate(a_pos, b_pos, energy, max_score, min_score):

    score_difference = max_score - min_score

    # If there are no remaining energy cells,
    # positional advantage is zero.
    if len(energy) == 0:
        positional_advantage = 0

    else:
        # Distance of A from its nearest remaining E
        distance_a = min(
            manhattan_distance(a_pos, e)
            for e in energy
        )

        # Distance of B from its nearest remaining E
        distance_b = min(
            manhattan_distance(b_pos, e)
            for e in energy
        )

        positional_advantage = distance_b - distance_a

    return score_difference + positional_advantage

### Evaluation Function

The evaluation function calculates the advantage of MAX.

It uses the difference between MAX and MIN scores and also considers the distance of both robots from the nearest remaining energy cell.

A smaller distance for A is better for MAX, while a smaller distance for B is better for MIN.

In [34]:
# Minimax algorithm
def minimax(a_pos, b_pos, energy, turn, depth,
            max_score, min_score, board, statistics):

    statistics["nodes_expanded"] += 1

    # Terminal / depth-limit state
    if depth == 0 or len(energy) == 0:
        return evaluate(
            a_pos, b_pos, energy, max_score, min_score
        ), None

    if turn == "MAX":

        possible_moves = generate_moves(a_pos, board)

        # No legal move -> terminal state
        if not possible_moves:
            return evaluate(
                a_pos, b_pos, energy, max_score, min_score
            ), None

        best_value = float("-inf")
        best_move = None

        for move_name, new_position in possible_moves:

            statistics["nodes_generated"] += 1

            new_energy = set(energy)
            new_max_score = max_score

            if new_position in new_energy:
                new_energy.remove(new_position)
                new_max_score += 10

            value, _ = minimax(
                new_position,
                b_pos,
                new_energy,
                "MIN",
                depth - 1,
                new_max_score,
                min_score,
                board,
                statistics
            )

            if value > best_value:
                best_value = value
                best_move = move_name

        return best_value, best_move

    else:

        possible_moves = generate_moves(b_pos, board)

        # No legal move -> terminal state
        if not possible_moves:
            return evaluate(
                a_pos, b_pos, energy, max_score, min_score
            ), None

        best_value = float("inf")
        best_move = None

        for move_name, new_position in possible_moves:

            statistics["nodes_generated"] += 1

            new_energy = set(energy)
            new_min_score = min_score

            if new_position in new_energy:
                new_energy.remove(new_position)
                new_min_score += 10

            value, _ = minimax(
                a_pos,
                new_position,
                new_energy,
                "MAX",
                depth - 1,
                max_score,
                new_min_score,
                board,
                statistics
            )

            if value < best_value:
                best_value = value
                best_move = move_name

        return best_value, best_move

### Minimax Algorithm

Here Minimax is used to find the best move for MAX.

MAX tries to select the move with the highest evaluation value.

MIN tries to select the move with the lowest evaluation value.

The search is limited by the given depth.

In [35]:
# Alpha-Beta pruning
def alpha_beta(a_pos, b_pos, energy, turn, depth,
               max_score, min_score, board,
               statistics, alpha, beta,
               heuristic_order=False):

    statistics["nodes_expanded"] += 1

    # Terminal / depth-limit state
    if depth == 0 or len(energy) == 0:
        return evaluate(
            a_pos, b_pos, energy, max_score, min_score
        ), None

    if turn == "MAX":

        possible_moves = generate_moves(a_pos, board)

        # No legal move -> terminal state
        if not possible_moves:
            return evaluate(
                a_pos, b_pos, energy, max_score, min_score
            ), None

        # Energy collecting moves first
        if heuristic_order:
            possible_moves.sort(
                key=lambda move: move[1] not in energy
            )

        best_value = float("-inf")
        best_move = None

        for i, (move_name, new_position) in enumerate(possible_moves):

            statistics["nodes_generated"] += 1

            new_energy = set(energy)
            new_max_score = max_score

            if new_position in new_energy:
                new_energy.remove(new_position)
                new_max_score += 10

            value, _ = alpha_beta(
                new_position,
                b_pos,
                new_energy,
                "MIN",
                depth - 1,
                new_max_score,
                min_score,
                board,
                statistics,
                alpha,
                beta,
                heuristic_order
            )

            if value > best_value:
                best_value = value
                best_move = move_name

            alpha = max(alpha, best_value)

            # Prune remaining moves
            if alpha >= beta:
                statistics["nodes_pruned"] += (
                    len(possible_moves) - i - 1
                )
                break

        return best_value, best_move

    else:

        possible_moves = generate_moves(b_pos, board)

        # No legal move -> terminal state
        if not possible_moves:
            return evaluate(
                a_pos, b_pos, energy, max_score, min_score
            ), None

        # Energy collecting moves first
        if heuristic_order:
            possible_moves.sort(
                key=lambda move: move[1] not in energy
            )

        best_value = float("inf")
        best_move = None

        for i, (move_name, new_position) in enumerate(possible_moves):

            statistics["nodes_generated"] += 1

            new_energy = set(energy)
            new_min_score = min_score

            if new_position in new_energy:
                new_energy.remove(new_position)
                new_min_score += 10

            value, _ = alpha_beta(
                a_pos,
                new_position,
                new_energy,
                "MAX",
                depth - 1,
                max_score,
                new_min_score,
                board,
                statistics,
                alpha,
                beta,
                heuristic_order
            )

            if value < best_value:
                best_value = value
                best_move = move_name

            beta = min(beta, best_value)

            # Prune remaining moves
            if alpha >= beta:
                statistics["nodes_pruned"] += (
                    len(possible_moves) - i - 1
                )
                break

        return best_value, best_move

### Alpha-Beta Pruning

Alpha-Beta pruning improves Minimax by avoiding branches that do not affect the final decision.

Alpha and Beta values are used to decide when a branch can be skipped.

This helps reduce the number of nodes explored.

In [36]:
# Run Minimax
def run_minimax(a_pos, b_pos, energy, depth, board):

    statistics = {
        "nodes_generated": 0,
        "nodes_expanded": 0,
        "nodes_pruned": 0
    }

    start_time = time.perf_counter()

    value, move = minimax(
        a_pos,
        b_pos,
        set(energy),
        "MAX",
        depth,
        0,
        0,
        board,
        statistics
    )

    end_time = time.perf_counter()

    return value, move, statistics, end_time - start_time

### Running Minimax

This function starts the Minimax search from the initial board.

It also records the number of generated nodes, expanded nodes and the execution time.

In [37]:
# Run Alpha-Beta
def run_alpha_beta(a_pos, b_pos, energy, depth,
                   board, heuristic_order):

    statistics = {
        "nodes_generated": 0,
        "nodes_expanded": 0,
        "nodes_pruned": 0
    }

    start_time = time.perf_counter()

    value, move = alpha_beta(
        a_pos,
        b_pos,
        set(energy),
        "MAX",
        depth,
        0,
        0,
        board,
        statistics,
        float("-inf"),
        float("inf"),
        heuristic_order
    )

    end_time = time.perf_counter()

    return value, move, statistics, end_time - start_time

### Running Alpha-Beta

This function runs Alpha-Beta pruning.

We can use it with normal move ordering or with energy collecting moves given priority.

The statistics and execution time are also recorded.

### Input

The given sample input is entered here.

The board has 5 rows and 7 columns, with A as MAX, B as MIN and three energy cells.

The search depth is 5.

In [38]:
# -----------------------------
# Main Program
# -----------------------------

rows, cols = map(int, input().split())

board = []

a_pos = None
b_pos = None
energy = set()

for r in range(rows):

    line = input().strip()
    board.append(line)

    for c in range(cols):

        if line[c] == 'A':
            a_pos = (r, c)

        elif line[c] == 'B':
            b_pos = (r, c)

        elif line[c] == 'E':
            energy.add((r, c))


player = input().strip()
depth = int(input())

# The assignment starts with MAX.ai
# We still read the input value as required.
if player != "MAX":
    print("Invalid starting player.")
    exit()

5 7
A.E.#..
.#..#E.
...#...
#E#..#.
....E.B
MAX
5


### Minimax Result

Now we run Minimax on the given input.

The program displays the best move, evaluation score, nodes generated, nodes expanded, search depth and execution time.

In [39]:
# -----------------------------
# Minimax
# -----------------------------

value1, move1, stats1, time1 = run_minimax(
    a_pos,
    b_pos,
    energy,
    depth,
    board
)

print()
print("========================================")
print("Algorithm: Minimax")
print("========================================")
print("Best Move:", move1)
print("Evaluation Score:", value1)
print("Nodes Generated:", stats1["nodes_generated"])
print("Nodes Expanded:", stats1["nodes_expanded"])
print("Search Depth:", depth)
print("Execution Time: {:.6f} seconds".format(time1))


Algorithm: Minimax
Best Move: RIGHT
Evaluation Score: 1
Nodes Generated: 66
Nodes Expanded: 67
Search Depth: 5
Execution Time: 0.000218 seconds


The best move found by Minimax is RIGHT.

The evaluation score is 1 and the search was performed up to depth 5.

### Alpha-Beta with Normal Move Ordering

Next, Alpha-Beta pruning is run using the normal move order:

UP → RIGHT → DOWN → LEFT.

The number of pruned nodes is also displayed.

In [40]:
# -----------------------------
# Alpha-Beta
# Normal ordering
# -----------------------------

value2, move2, stats2, time2 = run_alpha_beta(
    a_pos,
    b_pos,
    energy,
    depth,
    board,
    False
)

print()
print("========================================")
print("Algorithm: Alpha-Beta")
print("Move Ordering: Normal")
print("========================================")
print("Best Move:", move2)
print("Evaluation Score:", value2)
print("Nodes Generated:", stats2["nodes_generated"])
print("Nodes Expanded:", stats2["nodes_expanded"])
print("Nodes Pruned:", stats2["nodes_pruned"])
print("Search Depth:", depth)
print("Execution Time: {:.6f} seconds".format(time2))


Algorithm: Alpha-Beta
Move Ordering: Normal
Best Move: RIGHT
Evaluation Score: 1
Nodes Generated: 35
Nodes Expanded: 36
Nodes Pruned: 9
Search Depth: 5
Execution Time: 0.000171 seconds


Alpha-Beta also selects RIGHT as the best move.

Compared with Minimax, fewer nodes are expanded because some branches are pruned.

### Alpha-Beta with Heuristic Move Ordering

In this case, moves that collect an energy cell are considered first.

This is done to see whether better move ordering can improve Alpha-Beta pruning.

In [41]:
# -----------------------------
# Alpha-Beta
# Heuristic ordering
# -----------------------------

value3, move3, stats3, time3 = run_alpha_beta(
    a_pos,
    b_pos,
    energy,
    depth,
    board,
    True
)

print()
print("========================================")
print("Algorithm: Alpha-Beta")
print("Move Ordering: Energy First")
print("========================================")
print("Best Move:", move3)
print("Evaluation Score:", value3)
print("Nodes Generated:", stats3["nodes_generated"])
print("Nodes Expanded:", stats3["nodes_expanded"])
print("Nodes Pruned:", stats3["nodes_pruned"])
print("Search Depth:", depth)
print("Execution Time: {:.6f} seconds".format(time3))


Algorithm: Alpha-Beta
Move Ordering: Energy First
Best Move: RIGHT
Evaluation Score: 1
Nodes Generated: 35
Nodes Expanded: 36
Nodes Pruned: 9
Search Depth: 5
Execution Time: 0.000300 seconds


The best move is again RIGHT.

For this input, the number of expanded and pruned nodes is the same as normal move ordering.

### Comparison

Finally, we compare Minimax, Alpha-Beta with normal ordering and Alpha-Beta with energy-first ordering.

We compare their best move, evaluation score, expanded nodes, pruned nodes and execution time.

In [42]:
# -----------------------------
# Comparison
# -----------------------------

print()
print("========================================")
print("Comparison")
print("========================================")

print("Minimax:")
print("  Move =", move1)
print("  Evaluation =", value1)
print("  Expanded =", stats1["nodes_expanded"])
print("  Time = {:.6f} seconds".format(time1))

print()
print("Alpha-Beta Normal:")
print("  Move =", move2)
print("  Evaluation =", value2)
print("  Expanded =", stats2["nodes_expanded"])
print("  Pruned =", stats2["nodes_pruned"])
print("  Time = {:.6f} seconds".format(time2))

print()
print("Alpha-Beta Energy First:")
print("  Move =", move3)
print("  Evaluation =", value3)
print("  Expanded =", stats3["nodes_expanded"])
print("  Pruned =", stats3["nodes_pruned"])
print("  Time = {:.6f} seconds".format(time3))


Comparison
Minimax:
  Move = RIGHT
  Evaluation = 1
  Expanded = 67
  Time = 0.000218 seconds

Alpha-Beta Normal:
  Move = RIGHT
  Evaluation = 1
  Expanded = 36
  Pruned = 9
  Time = 0.000171 seconds

Alpha-Beta Energy First:
  Move = RIGHT
  Evaluation = 1
  Expanded = 36
  Pruned = 9
  Time = 0.000300 seconds


### Conclusion

All three methods selected RIGHT as the best move and gave the same evaluation score of 1.

Minimax expanded 67 nodes.

Alpha-Beta expanded only 36 nodes and pruned 9 nodes.

For this particular input, normal ordering and energy-first ordering produced the same number of expanded and pruned nodes.

So Alpha-Beta was able to reduce the number of nodes explored compared with Minimax.